In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
DATA_PATH = Path("../data/raw_credit_applications.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

print("Records:", len(raw))
print("Top-level type:", type(raw))
print("Keys in first record:", list(raw[0].keys()))

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw_credit_applications.json'

In [ ]:
df = pd.json_normalize(raw, sep=".")
print("Shape:", df.shape)
df.head(3)


Shape: (502, 21)


In [ ]:
def summarize_issue(mask: pd.Series, label: str) -> pd.DataFrame:
    n = int(mask.sum())
    total = int(mask.shape[0])
    pct = (n / total * 100) if total else 0.0
    return pd.DataFrame([{
        "issue": label,
        "affected_records": n,
        "percentage": round(pct, 2)
    }])

# Data Quality Issues

### Duplicate records


In [ ]:
df.columns.tolist()

['_id',
 'spending_behavior',
 'processing_timestamp',
 'applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.gender',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code',
 'financials.annual_income',
 'financials.credit_history_months',
 'financials.debt_to_income',
 'financials.savings_balance',
 'decision.loan_approved',
 'decision.rejection_reason',
 'loan_purpose',
 'decision.interest_rate',
 'decision.approved_amount',
 'financials.annual_salary',
 'notes']

Looking for Id duplicates

In [ ]:
def summarize_issue(mask: pd.Series, label: str) -> pd.DataFrame:
    n = int(mask.sum())
    total = int(mask.shape[0])
    pct = (n / total * 100) if total else 0.0
    return pd.DataFrame([{
        "issue": label,
        "affected_records": n,
        "percentage": round(pct, 2)
    }])

# 1) Reload RAW from file every time (so duplicates will always be detectable in this cell)
with DATA_PATH.open("r", encoding="utf-8") as f:
    raw_original = json.load(f)

df_original = pd.json_normalize(raw_original, sep=".")
rows_original = len(df_original)

# 2) Detect duplicated _id (system-level duplicates)
dup_mask = df_original["_id"].duplicated(keep=False)
dup_ids = df_original.loc[dup_mask, "_id"].unique()

summary_dup_ids = summarize_issue(dup_mask, "Duplicate application IDs (_id)")

print("=== DUPLICATE AUDIT (RAW) ===")
print("Rows in raw:", rows_original)
display(summary_dup_ids)

=== DUPLICATE AUDIT (RAW) ===
Rows in raw: 502


,issue,affected_records,percentage
0,Duplicate application IDs (_id),4,0.8


Id must be unique

In [ ]:
if len(dup_ids) > 0:
    dup_rows = df_original[df_original["_id"].isin(dup_ids)].sort_values("_id")
    print("\nDuplicated _id values:", list(dup_ids))
    display(dup_rows)
else:
    print("\nNo duplicated _id found in raw.")
    dup_rows = df_original.iloc[0:0].copy() 


Duplicated _id values: ['app_042', 'app_001']


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
383,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN
455,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION


We identified 2 duplicated application IDs (0.4%).

One case was caused by a duplicate ingestion error (explicitly labeled), while the other reflected a business resubmission.

In [ ]:
keep_idx = []
remove_idx = []

for app_id in dup_ids:
    g = df_original[df_original["_id"] == app_id]

    if (g["notes"] == "DUPLICATE_ENTRY_ERROR").any():
        idx_keep = g.isna().sum(axis=1).idxmin()      # most complete
    else:
        idx_keep = g.index.max()                       # latest (by index)

    keep_idx.append(idx_keep)
    remove_idx.extend([i for i in g.index if i != idx_keep])

In [ ]:
print("\n=== SELECTION (RAW) ===")
if len(dup_ids) > 0:
    df_keep = df_original.loc[keep_idx].sort_values("_id")
    df_remove = df_original.loc[remove_idx].sort_values("_id")

    print("Rows to KEEP (one per duplicated _id):")
    display(df_keep)

    print("Rows to REMOVE (duplicates):")
    display(df_remove)
else:
    print("Nothing to select: no duplicated _id in raw.")
    df_keep = df_original.iloc[0:0].copy()
    df_remove = df_original.iloc[0:0].copy()


=== SELECTION (RAW) ===
Rows to KEEP (one per duplicated _id):


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
383,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION


Rows to REMOVE (duplicates):


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
455,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


For ingestion errors we retained the most complete record to preserve information integrity. For resubmissions we retained the latest version to reflect application lifecycle logic.

In [ ]:
df_clean = df_original.drop(index=remove_idx).copy()

print("\n=== VERIFICATION (CLEAN) ===")
print("Rows after cleaning:", len(df_clean))
print("Rows removed:", len(df_original) - len(df_clean))
print("Remaining duplicated _id rows:", df_clean["_id"].duplicated(keep=False).sum())


if df_clean["_id"].duplicated(keep=False).any():
    display(df_clean[df_clean["_id"].duplicated(keep=False)].sort_values("_id"))


=== VERIFICATION (CLEAN) ===
Rows after cleaning: 500
Rows removed: 2
Remaining duplicated _id rows: 0


Looking for Social Security Number duplicates

In [ ]:
dup_ssn_mask_raw = (
    df_original["applicant_info.ssn"].duplicated(keep=False)
    & df_original["applicant_info.ssn"].notna()
)

display(summarize_issue(dup_ssn_mask_raw, "Business-level duplicates (duplicate SSN) in RAW"))

dup_ssns_raw = df_original.loc[dup_ssn_mask_raw, "applicant_info.ssn"].unique()

if len(dup_ssns_raw) > 0:
    print("Duplicated SSNs in RAW:", list(dup_ssns_raw))
    display(
        df_original[df_original["applicant_info.ssn"].isin(dup_ssns_raw)]
        .sort_values("applicant_info.ssn")[["_id", "applicant_info.full_name", "applicant_info.ssn", "notes"]]
    )
else:
    print("No duplicate SSNs found in RAW.")

,issue,affected_records,percentage
0,Business-level duplicates (duplicate SSN) in RAW,6,1.2


Duplicated SSNs in RAW: ['652-70-5530', '937-72-8731', '780-24-9300']


,_id,applicant_info.full_name,applicant_info.ssn,notes
8,app_042,Joseph Lopez,652-70-5530,NaN
354,app_042,Joseph Lopez,652-70-5530,RESUBMISSION
92,app_088,Susan Martinez,780-24-9300,NaN
122,app_016,Gary Wilson,780-24-9300,NaN
16,app_101,Sandra Smith,937-72-8731,NaN
499,app_234,Samuel Hill,937-72-8731,NaN


In [ ]:
dup_ssn_mask_raw = (
    df_original["applicant_info.ssn"].duplicated(keep=False)
    & df_original["applicant_info.ssn"].notna()
)

display(
    summarize_issue(
        dup_ssn_mask_raw,
        "Business-level duplicates (duplicate SSN) in RAW"
    )
)

dup_ssns_raw = df_original.loc[
    dup_ssn_mask_raw,
    "applicant_info.ssn"
].unique()

if len(dup_ssns_raw) > 0:
    print("Duplicated SSNs in RAW:", list(dup_ssns_raw))

    # Make sure all columns are visible
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)

    display(
        df_original[
            df_original["applicant_info.ssn"].isin(dup_ssns_raw)
        ].sort_values("applicant_info.ssn")
    )
else:
    print("No duplicate SSNs found in RAW.")

,issue,affected_records,percentage
0,Business-level duplicates (duplicate SSN) in RAW,6,1.2


Duplicated SSNs in RAW: ['652-70-5530', '937-72-8731', '780-24-9300']


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION
92,app_088,"[{'category': 'Transportation', 'amount': 339}]",NaN,Susan Martinez,susan.martinez83@protonmail.com,780-24-9300,172.23.224.13,F,1986-10-15,90229,55000,55,0.43,29118,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
122,app_016,"[{'category': 'Groceries', 'amount': 485}]",NaN,Gary Wilson,gary.wilson85@yahoo.com,780-24-9300,192.168.20.42,M,1959-12-11,10091,123000,71,0.37,86599,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
16,app_101,"[{'category': 'Groceries', 'amount': 527}, {'c...",NaN,Sandra Smith,sandra.smith99@icloud.com,937-72-8731,172.30.237.98,Female,1997-03-23,90255,55000,0,0.30,15364,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
499,app_234,"[{'category': 'Insurance', 'amount': 526}]",NaN,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,96000,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN


In [ ]:
# Start from dataset already cleaned for _id duplicates
df_final = df_clean.copy()

dup_ssn_mask = (
    df_final["applicant_info.ssn"].duplicated(keep=False)
    & df_final["applicant_info.ssn"].notna()
)

display(
    summarize_issue(
        dup_ssn_mask,
        "Business-level duplicates (duplicate SSN) BEFORE cleaning (on df_clean)"
    )
)

dup_ssns = df_final.loc[dup_ssn_mask, "applicant_info.ssn"].unique()

keep_idx, remove_idx = [], []

for ssn in dup_ssns:
    g = df_final[df_final["applicant_info.ssn"] == ssn]

    if (g["notes"] == "RESUBMISSION").any():
        idx_keep = g.index.max()                 # keep latest
    else:
        idx_keep = g.isna().sum(axis=1).idxmin() # keep most complete

    keep_idx.append(idx_keep)
    remove_idx.extend([i for i in g.index if i != idx_keep])

# Show what will be removed
if len(remove_idx) > 0:
    print("Rows to REMOVE for business-level duplicates (SSN):")
    display(
        df_final.loc[remove_idx][
            ["_id", "applicant_info.full_name", "applicant_info.ssn", "notes"]
        ]
    )
else:
    print("No business-level duplicates to remove.")

# Apply business-level cleaning
df_final = df_final.drop(index=remove_idx).copy()

print("After business cleaning, rows:", len(df_final))

print(
    "Remaining duplicated SSN rows:",
    (
        df_final["applicant_info.ssn"].duplicated(keep=False)
        & df_final["applicant_info.ssn"].notna()
    ).sum()
)

,issue,affected_records,percentage
0,Business-level duplicates (duplicate SSN) BEFO...,4,0.8


Rows to REMOVE for business-level duplicates (SSN):


,_id,applicant_info.full_name,applicant_info.ssn,notes
16,app_101,Sandra Smith,937-72-8731,NaN
122,app_016,Gary Wilson,780-24-9300,NaN


After business cleaning, rows: 498
Remaining duplicated SSN rows: 0


In the raw dataset, three SSNs appeared more than once (6 rows total). However, after removing _id duplicates, only two SSN groups remained duplicated (4 rows affected).

This difference occurs because one duplicated SSN belonged to records that were already resolved during the _id cleaning step.

To address business-level duplicates:

If a record was marked as RESUBMISSION, we retained the latest version.

Otherwise, we retained the most complete record (fewest missing values).

After applying this rule, no duplicated SSNs remain in the cleaned dataset.

In [ ]:
OUTPUT_PATH = Path("../data/credit_applications_clean_final.csv")
df_final.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)

Saved: ..\data\credit_applications_clean_final.csv


## Inconsistent data types across records

In [ ]:
df_original.dtypes

_id                                  object
spending_behavior                    object
processing_timestamp                 object
applicant_info.full_name             object
applicant_info.email                 object
applicant_info.ssn                   object
applicant_info.ip_address            object
applicant_info.gender                object
applicant_info.date_of_birth         object
applicant_info.zip_code              object
financials.annual_income             object
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.loan_approved                 bool
decision.rejection_reason            object
loan_purpose                         object
decision.interest_rate              float64
decision.approved_amount            float64
financials.annual_salary            float64
notes                                object
dtype: object

In [ ]:
type_summary = {}

for col in df_original.columns:
    types_in_col = df_original[col].dropna().map(type).unique()
    type_summary[col] = types_in_col

type_summary

{'_id': array([<class 'str'>], dtype=object),
 'spending_behavior': array([<class 'list'>], dtype=object),
 'processing_timestamp': array([<class 'str'>], dtype=object),
 'applicant_info.full_name': array([<class 'str'>], dtype=object),
 'applicant_info.email': array([<class 'str'>], dtype=object),
 'applicant_info.ssn': array([<class 'str'>], dtype=object),
 'applicant_info.ip_address': array([<class 'str'>], dtype=object),
 'applicant_info.gender': array([<class 'str'>], dtype=object),
 'applicant_info.date_of_birth': array([<class 'str'>], dtype=object),
 'applicant_info.zip_code': array([<class 'str'>], dtype=object),
 'financials.annual_income': array([<class 'int'>, <class 'str'>, <class 'float'>], dtype=object),
 'financials.credit_history_months': array([<class 'int'>], dtype=object),
 'financials.debt_to_income': array([<class 'float'>], dtype=object),
 'financials.savings_balance': array([<class 'int'>], dtype=object),
 'decision.loan_approved': array([<class 'bool'>], dtype=

In [5]:
mixed_type_cols = []

for col in df_original.columns:
    types_in_col = df_original[col].dropna().map(type).unique()
    if len(types_in_col) > 1:
        mixed_type_cols.append((col, types_in_col))

mixed_type_cols

NameError: name 'df_original' is not defined

In [ ]:
col = "financials.annual_income"

types = df_final[col].dropna().map(type).unique()
print("Types found:", types)
print("Mixed types?", len(types) > 1)

Types found: [<class 'float'>]
Mixed types? False


In [ ]:
if len(types) > 1:
    display(
        df_final[df_final[col].dropna().map(type) == str][["_id", col]].head(10)
    )

In [ ]:
missing_before = df_final[col].isna().sum()

df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

missing_after = df_final[col].isna().sum()

print("Missing before:", missing_before, "| Missing after:", missing_after)
print("Types after:", df_final[col].dropna().map(type).unique())

Missing before: 5 | Missing after: 5
Types after: [<class 'float'>]


The variable financials.annual_income contained inconsistent data types across records, including integers, floats, and numeric strings. Since this field represents a quantitative financial measure, it must be stored in a consistent numeric format. We standardized the column, converting all valid values to numeric format while preserving existing missing values. The number of missing observations remained unchanged (5 before and 5 after conversion), confirming that no additional data loss occurred during remediation. The column is now fully type-consistent and suitable for downstream analysis.

In [ ]:
df_final["processing_timestamp"] = pd.to_datetime(
    df_final["processing_timestamp"],
    errors="coerce"
)

In [ ]:
df_final[["processing_timestamp", "applicant_info.date_of_birth"]].dtypes

processing_timestamp            datetime64[ns, UTC]
applicant_info.date_of_birth         datetime64[ns]
dtype: object

The date fields were stored as strings and were converted to datetime format to ensure proper chronological operations, enable age and time calculations, and maintain type consistency for downstream analysis.

Confirm No Other Mixed-Type Columns Exist in Final Dataset:

In [ ]:
mixed_cols_final = []

for col in df_final.columns:
    types = df_final[col].dropna().map(type).unique()
    if len(types) > 1:
        mixed_cols_final.append((col, types))

mixed_cols_final

[]

## Missing or incomplete records

In [ ]:
missing_summary = (
    df_final.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df_final) * 100
).round(2)

missing_summary.sort_values("missing_percentage", ascending=False)

NameError: name 'df_final' is not defined

## Inconsistent coding/formatting of categorical fields

## Invalid or impossible values

## Inconsistent date formats